In [4]:
!pip install -q altair vl-convert-python pillow openpyxl

In [7]:
import os
import pandas as pd
import altair as alt
from PIL import Image

print('Altair', alt.__version__)
alt.data_transformers.disable_max_rows()

NEGRO, DORADO, ROJO = '#1B1B1B', '#D4AF37', '#B1121F'
DORADO_OSC, CHAMPAN, CREMA, GRIS = '#8A6D0B', '#E8D9A0', '#FBF6E9', '#555555'

ARCHIVO = 'Base de datos limpia original.xlsx'
if not os.path.exists(ARCHIVO):
    print('NO ENCUENTRO EL ARCHIVO. Archivos que sí hay en esta carpeta:', os.listdir('.'))
    raise FileNotFoundError(ARCHIVO)

df = pd.read_excel(ARCHIVO, sheet_name='base_limpia')
print('Filas y columnas:', df.shape)

def tipo_idioma(s):
    if s == 'Inglés':
        return 'Solo inglés'
    return 'Inglés y otro idioma' if 'Inglés' in s else 'Otro idioma'

df['tipo_idioma'] = df['idioma_actuacion'].map(tipo_idioma)
print(df['nominado_oscar_protagonico'].value_counts().to_dict())

Altair 5.5.0
Filas y columnas: (933, 12)
{'Sí': 470, 'No': 456, 'Pendiente': 7}


In [9]:
datos1 = df[df['nominado_oscar_protagonico'].isin(['Sí', 'No'])].copy()
datos1['nominado'] = (datos1['nominado_oscar_protagonico'] == 'Sí').astype(int)
n_pendientes = int((df['nominado_oscar_protagonico'] == 'Pendiente').sum())

tasa = (datos1.groupby('certamen')
              .agg(pct=('nominado', 'mean'), nominados=('nominado', 'sum'), n=('nominado', 'size'))
              .reset_index())

nombres = {'Festival de Cannes':  'Cannes · premio de interpretación',
           'Festival de Venecia': 'Venecia · Copa Volpi',
           'Festival de Berlín':  'Berlín · Oso de Plata (interpretación)',
           'BAFTA':               'BAFTA · interpretación protagónica',
           'Globo de Oro':        'Globo de Oro · interpretación protagónica'}
tasa['etiqueta'] = tasa['certamen'].map(nombres) + ' (n=' + tasa['n'].astype(str) + ')'
tasa['grupo'] = tasa['certamen'].map(lambda c: 'Globo de Oro y BAFTA' if c in ('Globo de Oro', 'BAFTA')
                                     else 'Festivales europeos')
tasa['pct'] = tasa['pct'].astype(float)
tasa['nominados'] = tasa['nominados'].astype(int)
tasa['n'] = tasa['n'].astype(int)

orden_certamenes = ['Festival de Cannes', 'Festival de Venecia', 'Festival de Berlín', 'BAFTA', 'Globo de Oro']
orden_etiquetas = [tasa.loc[tasa['certamen'] == c, 'etiqueta'].iloc[0] for c in orden_certamenes]

esperado = {'Festival de Cannes': 0.219, 'Festival de Venecia': 0.214, 'Festival de Berlín': 0.176,
            'BAFTA': 0.719, 'Globo de Oro': 0.792}
for c, v in esperado.items():
    obtenido = float(tasa.loc[tasa['certamen'] == c, 'pct'].iloc[0])
    assert abs(obtenido - v) < 0.002, (c, obtenido, v)
print('Cifras del gráfico 1 verificadas. Pendientes excluidos:', n_pendientes)

datos_g1 = tasa[['certamen', 'etiqueta', 'grupo', 'pct', 'nominados', 'n']]

base1 = alt.Chart(datos_g1).encode(
    y=alt.Y('etiqueta:N', sort=orden_etiquetas, title=None,
            axis=alt.Axis(labelLimit=420, labelFontSize=13)),
    x=alt.X('pct:Q', title='% de premiados que fueron nominados al Óscar como protagonistas',
            axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])))

barras1 = base1.mark_bar(size=34).encode(
    color=alt.Color('grupo:N',
                    scale=alt.Scale(domain=['Festivales europeos', 'Globo de Oro y BAFTA'],
                                    range=[DORADO_OSC, NEGRO]),
                    legend=alt.Legend(title=None, orient='top')),
    tooltip=[alt.Tooltip('certamen:N', title='Certamen'),
             alt.Tooltip('pct:Q', title='% nominado', format='.1%'),
             alt.Tooltip('nominados:Q', title='Nominados'),
             alt.Tooltip('n:Q', title='Premiados')])

etiquetas1 = base1.mark_text(align='left', dx=6, fontSize=14, fontWeight='bold').encode(
    text=alt.Text('pct:Q', format='.1%'))

grafico1 = (alt.layer(barras1, etiquetas1)
    .properties(width=620, height=300,
        title=alt.TitleParams(
            text='Premiados que fueron nominados al Óscar, por certamen',


            subtitle=['Se muestra una asociación, no una causa: no se puede afirmar que el premio provoque la nominación.',

                      'Fuente: base propia del equipo (1933–2026). Solo ganadores; se excluyen ' + str(n_pendientes) + ' casos 2026 pendientes.',

                      'Nominación como protagonista (mejor actor o actriz). Datos con verificaciones pendientes.'],
            anchor='start', fontSize=20, subtitleFontSize=11, subtitleColor=GRIS))
    .configure(background=CREMA)
    .configure_view(stroke=None))

grafico1

Cifras del gráfico 1 verificadas. Pendientes excluidos: 7


alt.LayerChart(...)

In [10]:
def exportar(grafico, nombre):
    grafico.save(nombre + '.html')
    print('HTML listo:', nombre + '.html')
    try:
        grafico.save(nombre + '.png', scale_factor=2.0)
        Image.open(nombre + '.png').convert('RGB').save(nombre + '.jpg', quality=95)
        print('JPG listo:', nombre + '.jpg')
    except Exception as e:
        print('No se pudo crear el JPG automáticamente.', type(e).__name__, str(e)[:200])
        print('Alternativa: en el gráfico, haz clic en los tres puntos (arriba a la derecha) > Save as PNG.')

exportar(grafico1, 'grafico_1')

HTML listo: grafico_1.html
JPG listo: grafico_1.jpg


In [11]:
orden_tipos = ['Otro idioma', 'Inglés y otro idioma', 'Solo inglés']

conteo = df.groupby(['certamen', 'tipo_idioma']).size().rename('personas').reset_index()
completo = pd.MultiIndex.from_product([orden_certamenes, orden_tipos], names=['certamen', 'tipo_idioma'])
conteo = conteo.set_index(['certamen', 'tipo_idioma']).reindex(completo).fillna(0).reset_index()
conteo['personas'] = conteo['personas'].astype(int)
conteo['n_certamen'] = conteo.groupby('certamen')['personas'].transform('sum').astype(int)
conteo['pct'] = (conteo['personas'] / conteo['n_certamen']).astype(float)

conteo['tipo_idioma'] = pd.Categorical(conteo['tipo_idioma'], categories=orden_tipos, ordered=True)
conteo = conteo.sort_values(['certamen', 'tipo_idioma']).reset_index(drop=True)
conteo['y1'] = conteo.groupby('certamen')['pct'].cumsum()
conteo['y0'] = conteo['y1'] - conteo['pct']
conteo['ymid'] = (conteo['y0'] + conteo['y1']) / 2
conteo['tipo_idioma'] = conteo['tipo_idioma'].astype(str)

nombres_cortos = {'Festival de Cannes': 'Cannes', 'Festival de Venecia': 'Venecia',
                  'Festival de Berlín': 'Berlín', 'BAFTA': 'BAFTA', 'Globo de Oro': 'Globo de Oro'}
conteo['certamen_n'] = conteo['certamen'].map(nombres_cortos) + ' (n=' + conteo['n_certamen'].astype(str) + ')'
orden_x = [nombres_cortos[c] + ' (n=' + str(int(conteo.loc[conteo['certamen'] == c, 'n_certamen'].iloc[0])) + ')'
           for c in orden_certamenes]
conteo['color_texto'] = conteo['tipo_idioma'].map(
    lambda t: NEGRO if t == 'Inglés y otro idioma' else '#FFFFFF')

def p(c, t):
    return float(conteo[(conteo['certamen'] == c) & (conteo['tipo_idioma'] == t)]['pct'].iloc[0])
assert abs(p('Festival de Cannes', 'Otro idioma') - 0.584) < 0.002
assert abs(p('BAFTA', 'Solo inglés') - 0.910) < 0.002
assert abs(p('Globo de Oro', 'Solo inglés') - 0.950) < 0.002
print('Cifras del gráfico 2 verificadas')

datos_g2 = conteo[['certamen', 'certamen_n', 'tipo_idioma', 'personas', 'pct', 'y0', 'y1', 'ymid', 'color_texto']]
datos_rotulos = datos_g2[datos_g2['pct'] >= 0.035]

colores_tipo = alt.Scale(domain=orden_tipos, range=[ROJO, DORADO, NEGRO])

barras2 = alt.Chart(datos_g2).mark_bar(size=62).encode(
    x=alt.X('certamen_n:N', sort=orden_x, title=None,
            axis=alt.Axis(labelAngle=0, labelFontSize=12, labelLimit=150)),
    y=alt.Y('y0:Q', title='Proporción de las actuaciones premiadas',
            axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])),
    y2=alt.Y2('y1'),
    color=alt.Color('tipo_idioma:N', scale=colores_tipo,
                    legend=alt.Legend(title='Idioma de la actuación', orient='right')),
    tooltip=[alt.Tooltip('certamen:N', title='Certamen'),
             alt.Tooltip('tipo_idioma:N', title='Idioma'),
             alt.Tooltip('pct:Q', title='Proporción', format='.1%'),
             alt.Tooltip('personas:Q', title='Personas')])

rotulos = alt.Chart(datos_rotulos).mark_text(fontSize=13, fontWeight='bold').encode(
    x=alt.X('certamen_n:N', sort=orden_x),
    y=alt.Y('ymid:Q'),
    text=alt.Text('pct:Q', format='.0%'),
    color=alt.Color('color_texto:N', scale=None, legend=None))

grafico2 = (alt.layer(barras2, rotulos)
    .properties(width=640, height=420,
        title=alt.TitleParams(
            text='Idioma de las actuaciones premiadas, por certamen',
            subtitle=['Cada barra suma 100%. No se rotulan los tramos menores a 3,5%.',
                      'Fuente: base propia del equipo (933 premios de actuación, 1933–2026). Solo ganadores.',
                      'El idioma es el de la actuación premiada, no el de toda la película.'],
            anchor='start', fontSize=20, subtitleFontSize=11, subtitleColor=GRIS))
    .configure(background=CREMA)
    .configure_view(stroke=None))

grafico2

Cifras del gráfico 2 verificadas


alt.LayerChart(...)

In [12]:
exportar(grafico2, 'grafico_2')

HTML listo: grafico_2.html
JPG listo: grafico_2.jpg
